In [1]:
import pandas as pd
import numpy as np
import requests
import os
import dotenv
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import plotly.colors as colors
import textwrap
from plotly.subplots import make_subplots
from datetime import datetime
from scipy.stats import gaussian_kde



In [2]:
dotenv.load_dotenv(override=True)

True

In [3]:
token = os.getenv("REDCAP_API_KEY")

In [4]:
# id_list=[
#     "79-1", "79-2", "79-4", "79-6", "79-7", "79-8", "79-10", "79-11",
#     "105-1", "105-5", "105-6", "105-7",
#     "99-1", "99-2", "99-3",
#     "106-1", "106-2", "106-3", "106-4", "106-5", "106-6", "106-7",
#     "78-1", "78-2", "78-3", "78-4", "78-6", "78-8",
#     "101-1", "101-4", "101-5", "101-7", "101-8", "101-9", "101-10", "101-11",
#     "100-1", "100-2", "100-4", "100-6", "100-8", "100-9", "100-13"
# ]

In [5]:
data = {
    'token': token,
    'content': 'record',
    'action': 'export',
    'format': 'json',
    'type': 'eav',
    # 'records[]': id_list,  # Passa a lista diretamente aqui
    'fields[]': ['record_id'],
    'forms[]': ['demografia'],
    'events[]': ['visita_1__screenin_arm_1'],
    'rawOrLabel': 'label',
    'rawOrLabelHeaders': 'label',
    'exportCheckboxLabel': 'false',
    'exportSurveyFields': 'false',
    'exportDataAccessGroups': 'false',
    'returnFormat': 'json'
}

In [6]:
r = requests.post('https://svriglobal.org/redcap/api/', data=data)
print('HTTP Status:', r.status_code)
print(r.json())

HTTP Status: 200
[{'record': '56-1', 'redcap_event_name': 'Visita 1 - Screening', 'field_name': 'demografia_centro', 'value': 'Hospital e Maternidade Dr. Christóvão da Gama S.A'}, {'record': '56-1', 'redcap_event_name': 'Visita 1 - Screening', 'field_name': 'demografia_complete', 'value': 'Complete'}, {'record': '56-1', 'redcap_event_name': 'Visita 1 - Screening', 'field_name': 'demografia_data_nascimento', 'value': '2005-02-11'}, {'record': '56-1', 'redcap_event_name': 'Visita 1 - Screening', 'field_name': 'demografia_dt_visita', 'value': '2025-12-16'}, {'record': '56-1', 'redcap_event_name': 'Visita 1 - Screening', 'field_name': 'demografia_escolaridade', 'value': 'Ensino Médio Completo'}, {'record': '56-1', 'redcap_event_name': 'Visita 1 - Screening', 'field_name': 'demografia_etnia', 'value': 'Hispânico ou Latino'}, {'record': '56-1', 'redcap_event_name': 'Visita 1 - Screening', 'field_name': 'demografia_iniciais', 'value': 'WBL'}, {'record': '56-1', 'redcap_event_name': 'Visita 1 

In [7]:
df = pd.DataFrame(r.json())

In [8]:
# filtrar df onde field_name = demografia_centro
distribuicao_sites = df[df['field_name'] == 'demografia_centro']

In [9]:
# contar a distribuição de sites
distribuicao_sites_count = distribuicao_sites['value'].value_counts()
print(distribuicao_sites_count)

value
Centro de Oncologia do Paraná                                 21
Hospital e Maternidade Dr. Christóvão da Gama S.A             19
Irmandade da Santa Casa de Misericórdia de Santos             19
Clinica de Ortopedia Campo Largo S/S LTDA                     15
Centro de Investigação Clínica Ciências Médicas (CIC CMMG)     4
Name: count, dtype: int64


In [10]:
# Gráfico de barras em plotly
def quebrar_texto(texto, largura_max=16):
    """
    Quebra o texto inserindo <br> para manter cada linha 
    com aproximadamente 'largura_max' caracteres.
    """
    return "<br>".join(textwrap.wrap(str(texto), width=largura_max))
sites_formatados = [quebrar_texto(site, largura_max=18) for site in distribuicao_sites_count.index]
    
fig = px.bar(
    distribuicao_sites_count, 
    x=sites_formatados, 
    y=distribuicao_sites_count.values, 
    labels={'x':'', 'y':''}, 
    title='Distribuição de Sites'
)
# adicionar o valor no lado externo superior da barra, limpar o fundo e tirar a grade
fig.update_traces(texttemplate='%{y}', textposition='inside')
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False)
)
#Retirar os valores do eixo Y (somente exibir os valores nas barras)
fig.update_yaxes(showticklabels=False)
fig.show()

In [11]:
# filtrar df onde field_name = demografia_sexo
distribuicao_sexo = df[df['field_name'] == 'demografia_sexo']

In [12]:
# contar a distribuição de sites
distribuicao_sexo_count = distribuicao_sexo['value'].value_counts()
print(distribuicao_sexo_count)

value
Masculino    63
Feminino     17
Name: count, dtype: int64


In [13]:
# contar percentual a distribuição de gênero
distribuicao_sexo_percentual = distribuicao_sexo_count / distribuicao_sexo_count.sum() * 100
print(distribuicao_sexo_percentual)

value
Masculino    78.75
Feminino     21.25
Name: count, dtype: float64


In [14]:
# Gráfico de pizza em plotly
fig = px.pie(
    distribuicao_sexo_count, 
    names=distribuicao_sexo_count.index, 
    values=distribuicao_sexo_count.values, 
    title='Distribuição de Gênero/Sexo'
)
# adicionar o valor dentro das fatias, limpar o fundo e tirar a grade
fig.update_traces(texttemplate='%{value}', textposition='inside')
fig.update_layout(
    plot_bgcolor='white'
)
fig.show()

In [15]:
# Gráfico de pizza em plotly com valores relativos
fig = px.pie(
    distribuicao_sexo_percentual, 
    names=distribuicao_sexo_percentual.index, 
    values=distribuicao_sexo_percentual.values, 
    title='Distribuição de Gênero/Sexo (Percentual)'
)
# adicionar o valor dentro das fatias, limpar o fundo e tirar a grade
fig.update_traces(texttemplate='%{value:.2f}%', textposition='inside')
fig.update_layout(
    plot_bgcolor='white'
)
fig.show()

<h1>Idade</h1>

In [16]:
distribuicao_por_idade = df[df['field_name'] == 'demografia_data_nascimento']

In [17]:
# Excluir linhas com value = 'NA'
distribuicao_por_idade = distribuicao_por_idade.dropna(subset=['value'])
distribuicao_por_idade = distribuicao_por_idade[distribuicao_por_idade['value'] != 'NA']

# Converter a coluna 'value' para datetime
distribuicao_por_idade['value'] = pd.to_datetime(distribuicao_por_idade['value'])
# alterar o nome da coluna para dt_nasc
distribuicao_por_idade = distribuicao_por_idade.rename(columns={'value': 'dt_nasc'})



In [18]:
data_atual = datetime.now()
distribuicao_por_idade['idade'] = distribuicao_por_idade['dt_nasc'].apply(lambda x: data_atual.year - x.year - ((data_atual.month, data_atual.day) < (x.month, x.day)))
distribuicao_por_idade = distribuicao_por_idade[[
    'record',
    'idade'
]]

In [19]:
# Cálculo da idade mínima arredondada para baixo para o múltiplo de 10 mais próximo
idade_minima = (min(distribuicao_por_idade['idade'])//10) *10
# Cálculo da idade máxima arredondada para cima para o múltiplo de 10 mais próximo
idade_max = ((max(distribuicao_por_idade['idade'])//10)+1) * 10
# Criação dos bins de 10 em 10 anos
bins = list(range(idade_minima, idade_max + 10, 10))
# Criação dos rótulos para cada bin de idade
labels = [f'{bins[i]}-{bins[i+1]-1}' for i in range(len(bins)-1)]
print(labels)

['10-19', '20-29', '30-39', '40-49', '50-59']


In [20]:
distribuicao_por_faixa_etaria = distribuicao_por_idade.copy()
distribuicao_por_faixa_etaria['faixa_etaria'] = pd.cut(distribuicao_por_faixa_etaria['idade'],bins=bins, labels=labels,right=False)
distribuicao_por_faixa_etaria=distribuicao_por_faixa_etaria.value_counts().reset_index()
distribuicao_por_faixa_etaria.columns = ['index', 'idade', 'faixa_etaria', 'count']

In [21]:

distribuicao_por_faixa_etaria=distribuicao_por_faixa_etaria[['faixa_etaria', 'count']]
#Excluir linhas com count = 0
distribuicao_por_faixa_etaria = distribuicao_por_faixa_etaria[distribuicao_por_faixa_etaria['count'] != 0]


In [22]:
fig = px.histogram(
    distribuicao_por_faixa_etaria,
    x='faixa_etaria',
    title='Distribuição de Idade dos Participantes',
    labels=labels,
    color_discrete_sequence=px.colors.qualitative.Set3,
    nbins=len(labels),
    category_orders={'faixa_etaria': labels},
    text_auto = True
)

fig.update_layout(
    yaxis_title='Frequência',
    bargap=0,
    plot_bgcolor='white',
)
fig.update_traces(
    marker_line_width=1.5,
    marker_line_color='black'
)



fig.show()

<h1>Raça</h1>

In [23]:
distribuicao_por_raca = df[df['field_name'] == 'demografia_raca']

In [24]:
distribuicao_pop_por_raca = distribuicao_por_raca.copy()
distribuicao_pop_por_raca = distribuicao_pop_por_raca['value'].value_counts()
print(distribuicao_pop_por_raca)

value
Branco             49
Morena ou Parda    22
Negra ou Preta      9
Name: count, dtype: int64


In [25]:
distribuicao_relativa_pop_por_raca = distribuicao_pop_por_raca / distribuicao_pop_por_raca.sum()*100
#imprimir no formato xxx.xx%
print(distribuicao_relativa_pop_por_raca.map(lambda percentual: f"{percentual:.2f}%"))

value
Branco             61.25%
Morena ou Parda    27.50%
Negra ou Preta     11.25%
Name: count, dtype: str


In [26]:
fig = px.bar(
    x=distribuicao_pop_por_raca.index,
    y=distribuicao_pop_por_raca,
    text=distribuicao_pop_por_raca
)
fig.show()

In [27]:
fig = px.pie(
    names = distribuicao_relativa_pop_por_raca.index,
    values = distribuicao_relativa_pop_por_raca.values,
    title='Distribuição de Raça (Percentual)'
)
# adicionar o valor dentro das fatias, limpar o fundo e tirar a grade
fig.update_traces(texttemplate='%{value:.2f}%', textposition='inside')
fig.update_layout(
    plot_bgcolor='white'
)
fig.show()

<h1>Etnia</h1>

In [28]:
distribuicao_por_etnia = df[df['field_name'] == 'demografia_etnia']

In [29]:
distribuicao_por_etnia = distribuicao_por_etnia[[
    'record',
    'value'
]]
distribuicao_por_etnia_abs = distribuicao_por_etnia.copy()
distribuicao_por_etnia_abs = distribuicao_por_etnia_abs['value'].value_counts()
distribuicao_por_etnia_abs

value
Hispânico ou Latino        72
Não hispânico ou Latino     8
Name: count, dtype: int64

In [30]:
# Gráfico de pizza em plotly
fig = px.pie(
    distribuicao_por_etnia_abs, 
    names=distribuicao_por_etnia_abs.index, 
    values=distribuicao_por_etnia_abs.values, 
    title='Distribuição por Etnia'
)
# adicionar o valor dentro das fatias, limpar o fundo e tirar a grade
fig.update_traces(texttemplate='%{value}', textposition='inside')
fig.update_layout(
    plot_bgcolor='white'
)
fig.show()

In [31]:
distribuicao_por_etnia_relativa = distribuicao_por_etnia.copy()
distribuicao_por_etnia_relativa = distribuicao_por_etnia_relativa['value'].value_counts(normalize=True)*100
distribuicao_por_etnia_relativa.map(lambda percentual: f"{percentual:.2f}%")


value
Hispânico ou Latino        90.00%
Não hispânico ou Latino    10.00%
Name: proportion, dtype: str

In [32]:
fig = px.pie(
    names = distribuicao_por_etnia_relativa.index,
    values = distribuicao_por_etnia_relativa.values,
    title='Distribuição relativa por etnia'
)
# adicionar o valor dentro das fatias, limpar o fundo e tirar a grade
fig.update_traces(texttemplate='%{value:.2f}%', textposition='inside')
fig.update_layout(
    plot_bgcolor='white'
)
fig.show()

<h1>Escolaridade</h1>

In [33]:
distribuicao_por_escolaridade = df[df['field_name'] == 'demografia_escolaridade']

In [34]:
distribuicao_por_escolaridade = distribuicao_por_escolaridade[[
    'field_name',
    'value'
]]

In [35]:
distribuicao_por_escolaridade_abs = distribuicao_por_escolaridade['value'].value_counts()
distribuicao_por_escolaridade_relativa = distribuicao_por_escolaridade['value'].value_counts(normalize=True)*100
print(distribuicao_por_escolaridade_abs)
print(distribuicao_por_escolaridade_relativa.map(lambda percentual: f"{percentual:.2f}%"))

value
Ensino Médio Completo            41
Ensino Superior Completo         16
Ensino Superior Incompleto       10
Ensino Fundamental Incompleto     4
Ensino Fundamental Completo       4
Ensino Médio Incompleto           3
NA                                2
Name: count, dtype: int64
value
Ensino Médio Completo            51.25%
Ensino Superior Completo         20.00%
Ensino Superior Incompleto       12.50%
Ensino Fundamental Incompleto     5.00%
Ensino Fundamental Completo       5.00%
Ensino Médio Incompleto           3.75%
NA                                2.50%
Name: proportion, dtype: str


In [36]:
escolaridade_formatado = [quebrar_texto(escolaridade, largura_max=18) for escolaridade in distribuicao_por_escolaridade_abs.index]
fig = px.bar(
    data_frame=distribuicao_por_escolaridade_abs,
    x=escolaridade_formatado,
    y=distribuicao_por_escolaridade_abs.values
)
fig.update_layout(
    title="Distribuição por Escolaridade",
    xaxis_title="Escolaridade",
    yaxis_title="Frequência",
    plot_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False)
)
fig.update_traces(
    marker_color='blue',
    texttemplate='%{y}',
    textposition='inside'
)
fig.show()

In [37]:
distribuicao_por_escolaridade_relativa

value
Ensino Médio Completo            51.25
Ensino Superior Completo         20.00
Ensino Superior Incompleto       12.50
Ensino Fundamental Incompleto     5.00
Ensino Fundamental Completo       5.00
Ensino Médio Incompleto           3.75
NA                                2.50
Name: proportion, dtype: float64

In [38]:
fig = px.pie(
    names = distribuicao_por_escolaridade_relativa.index,
    values = distribuicao_por_escolaridade_relativa.values,
    title='Distribuição relativa por escolaridade'
)
# adicionar o valor dentro das fatias, limpar o fundo e tirar a grade
fig.update_traces(texttemplate='%{value:.2f}%', textposition='inside')
fig.update_layout(
    plot_bgcolor='white'
)
fig.show()

In [39]:
# import pandas as pd
# import numpy as np
# import plotly.express as px

# # 1. Dados originais simulados a partir do seu print
# data = {
#     'Escolaridade': [
#         'Ensino Médio Completo', 'Ensino Superior Completo', 'Ensino Superior Incompleto', 
#         'Ensino Fundamental Incompleto', 'Ensino Fundamental Completo', 'Ensino Médio Incompleto', 'NA'
#     ],
#     'Percentual': [51.90, 18.99, 12.66, 5.06, 5.06, 3.80, 2.53]
# }
# df_escolaridade = pd.DataFrame(data)
df_escolaridade = pd.DataFrame(distribuicao_por_escolaridade_relativa).reset_index()

# 2. Arredondar os percentuais para números inteiros (cada 1% = 1 quadrado)
df_escolaridade['Quadrados'] = df_escolaridade['proportion'].round().astype(int)

# Corrigir possíveis erros de arredondamento para garantir que a soma seja exatamente 100
diferenca = 100 - df_escolaridade['Quadrados'].sum()
if diferenca != 0:
    # Adiciona ou subtrai a diferença da maior categoria para fechar 100
    df_escolaridade.loc[df_escolaridade['Quadrados'].idxmax(), 'Quadrados'] += diferenca

# 3. Criar uma lista com as categorias repetidas com base no número de quadrados
categorias_waffle = []
for index, row in df_escolaridade.iterrows():
    categorias_waffle.extend([row["value"]] * row['Quadrados'])

# 4. Construir o grid 10x10 (coordenadas X e Y para cada quadradinho)
# np.meshgrid cria a matriz, e o flatten() transforma em uma lista contínua
x, y = np.meshgrid(np.arange(10), np.arange(10))
waffle_df_escolaridade = pd.DataFrame({
    'x': x.flatten(),
    'y': y.flatten(),
    'value': categorias_waffle
})

# 5. Plotar usando px.scatter
fig = px.scatter(
    waffle_df_escolaridade, 
    x='x', 
    y='y', 
    color='value',
    title='Distribuição relativa por escolaridade',
    hover_name='value'
)

# 6. Formatação de Waffle: Mudar o marcador para quadrado, ajustar tamanho e remover eixos
fig.update_traces(
    marker=dict(
        symbol='square', 
        size=28, # Tamanho do quadrado (ajuste conforme o tamanho do seu layout)
        line=dict(width=0.5, color='white'), # Borda branca para separar os quadrados
        
    ),
    hovertemplate="<b>%{hovertext}</b><extra></extra>" # Limpa o tooltip para não mostrar 'x' e 'y'
)

fig.update_layout(
    xaxis=dict(showgrid=False, zeroline=False, visible=False), # Oculta eixo X
    yaxis=dict(showgrid=False, zeroline=False, visible=False), # Oculta eixo Y
    plot_bgcolor='white',
    height=450,
    width=670,
    legend=dict(
        title="<b>Escolaridade</b>",
        yanchor="middle",
        y=0.5,
        xanchor="left",
        x=1.05
    )
)

fig.show()

<h1>Inclusão</h1>

In [40]:
data = {
    'token': token,
    'content': 'record',
    'action': 'export',
    'format': 'json',
    'type': 'eav',
    # 'records[]': id_list,  # Passa a lista diretamente aqui
    'fields[]': ['record_id'],
    'forms[0]': 'demografia',
    'forms[1]': 'status_do_participante_de_pesquisa_no_estudo',
    'events[0]': 'status_do_particip_arm_1',
    'events[1]': 'visita_1__screenin_arm_1',
    'rawOrLabel': 'label',
    'rawOrLabelHeaders': 'label',
    'exportCheckboxLabel': 'false',
    'exportSurveyFields': 'false',
    'exportDataAccessGroups': 'false',
    'returnFormat': 'json'
}
r = requests.post('https://svriglobal.org/redcap/api/', data=data)
print('HTTP Status:', r.status_code)
print(r.json())

HTTP Status: 200
[{'record': '56-1', 'redcap_event_name': 'Status do Participante', 'redcap_repeat_instrument': '', 'redcap_repeat_instance': 1, 'field_name': 'record_id', 'value': '56-1'}, {'record': '56-1', 'redcap_event_name': 'Status do Participante', 'redcap_repeat_instrument': '', 'redcap_repeat_instance': 1, 'field_name': 'status_instrumento', 'value': 'Status do Participante de Pesquisa no Estudo'}, {'record': '56-1', 'redcap_event_name': 'Status do Participante', 'redcap_repeat_instrument': '', 'redcap_repeat_instance': 1, 'field_name': 'status_pp_dt_visita', 'value': '2025-12-19'}, {'record': '56-1', 'redcap_event_name': 'Status do Participante', 'redcap_repeat_instrument': '', 'redcap_repeat_instance': 1, 'field_name': 'status_pp_status', 'value': 'Falha de seleção'}, {'record': '56-1', 'redcap_event_name': 'Status do Participante', 'redcap_repeat_instrument': '', 'redcap_repeat_instance': 1, 'field_name': 'status_pp_motivo', 'value': 'Solicitação do participante.'}, {'recor

In [41]:
df_inclusao = pd.DataFrame(r.json())

In [42]:
# Filtrar de df_inclusao os seguintes campos da variável [field_name]
field_name_list = [
    'record_id',
    'status_pp_status',
    'demografia_dt_visita'
]

df_inclusao_dt_filtered = df_inclusao[df_inclusao['field_name'].isin(field_name_list)]
df_inclusao_dt_filtered.drop(columns=['redcap_event_name', 'redcap_repeat_instrument', 'redcap_repeat_instance'], inplace=True)
# drop duplicates
df_inclusao_dt_filtered = df_inclusao_dt_filtered.drop_duplicates()
#excluir a linha em que field_name == record_id
df_inclusao_dt_filtered = df_inclusao_dt_filtered[df_inclusao_dt_filtered['field_name'] != 'record_id']


In [43]:
df_inclusao_dt_filtered = df_inclusao_dt_filtered.pivot_table(
    index = ['record'],
    columns = 'field_name',
    values = 'value',
    aggfunc = 'first'
).reset_index()
df_inclusao_dt_filtered.head()


field_name,record,demografia_dt_visita,status_pp_status
0,100-1,2026-03-20,Falha de seleção
1,100-10,2025-05-11,Ativo
2,100-11,2026-05-11,Ativo
3,100-12,2026-05-25,Falha de seleção
4,100-13,2026-06-05,Ativo


In [44]:
# tipificando record e status_pp_status como "texto" e demografia_dt_visita como datetime
df_inclusao_dt_filtered['record'] = df_inclusao_dt_filtered['record'].astype(str)
df_inclusao_dt_filtered['status_pp_status'] = df_inclusao_dt_filtered['status_pp_status'].astype(str)
df_inclusao_dt_filtered['demografia_dt_visita'] = pd.to_datetime(df_inclusao_dt_filtered['demografia_dt_visita'])

In [45]:
# Gráfico de linhas para análise temporal de inclusão de participantes com plotly

df_inclusao_dt_filtered['mes_inclusao'] = df_inclusao_dt_filtered['demografia_dt_visita'].dt.to_period('M').dt.to_timestamp()
df_contagem = df_inclusao_dt_filtered.groupby('mes_inclusao').size().reset_index(name='qtd_inclusao')

fig_inclusao_mensal = px.line(
    df_contagem,
    x='mes_inclusao',
    y='qtd_inclusao',
    title='Análise temporal de inclusão de participantes por mês',
    labels={'mes_inclusao': 'Mês da Visita', 'qtd_inclusao': 'Número de Pacientes'},
    # text_auto=True # Mostra o número em cima da barra
)
fig_inclusao_mensal.update_xaxes(dtick="M1", tickformat="%b\n%Y")
fig_inclusao_mensal.show()

In [46]:
df_ordenado = df_inclusao_dt_filtered.sort_values(by='demografia_dt_visita').copy()
df_ordenado['contagem_individual'] = 1
df_ordenado['inclusao_cumulativa'] = df_ordenado['contagem_individual'].cumsum()
fig_cumulativa = px.area(
    df_ordenado,
    x='demografia_dt_visita',
    y='inclusao_cumulativa',
    title='Inclusão cumulativa de participantes',
    # color = 'status_pp_status'
)
fig_cumulativa.show()


<h1>Distribuição por status</h1>

In [47]:
df_status_pp = df_ordenado.copy()
df_status_pp = df_status_pp[[
    'record',
    'status_pp_status'
]]

df_status_pp.rename(columns={'status_pp_status': 'status'}, inplace=True)
df_status_pp

field_name,record,status
17,100-6,Ativo
1,100-10,Ativo
56,57-1,Concluído
55,56-1,Falha de seleção
65,79-1,Concluído
...,...,...
79,99-4,Triagem
13,100-21,Triagem
46,106-15,Triagem
45,106-14,Triagem


In [48]:
distribuicao_status_abs = df_status_pp['status'].value_counts()
distribuicao_status_abs

status
Ativo                  33
Concluído              17
Falha de seleção       17
Triagem                 7
Perda de seguimento     4
Retirada de TCLE        2
Name: count, dtype: int64

In [49]:
distribuicao_status_relativa = df_status_pp['status'].value_counts(normalize=True)*100
distribuicao_status_relativa

status
Ativo                  41.25
Concluído              21.25
Falha de seleção       21.25
Triagem                 8.75
Perda de seguimento     5.00
Retirada de TCLE        2.50
Name: proportion, dtype: float64

In [50]:
fig = px.bar(
    data_frame = df_status_pp,
    x='status',
    y='record',
    
)
fig.show()



In [51]:

status_formatados = [quebrar_texto(site, largura_max=18) for site in distribuicao_status_abs.index]
    
fig = px.bar(
    distribuicao_status_abs, 
    x=status_formatados, 
    y=distribuicao_status_abs.values, 
    labels={'x':'', 'y':''}, 
    title='Distribuição de Status'
)
# adicionar o valor no lado externo superior da barra, limpar o fundo e tirar a grade
fig.update_traces(texttemplate='%{y}', textposition='inside')
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False)
)
#Retirar os valores do eixo Y (somente exibir os valores nas barras)
fig.update_yaxes(showticklabels=False)
fig.show()

<h1>Antropometria</h1>

In [52]:
data = {
    'token': token,
    'content': 'record',
    'action': 'export',
    'format': 'json',
    'type': 'eav',
    # 'records[]': id_list,  # Passa a lista diretamente aqui
    'fields[0]': 'form_medico_ex_fisico_peso',
    'fields[1]': 'form_medico_ex_fisico_alt',
    'fields[2]': 'form_medico_ex_fisico_q4',
    'rawOrLabel': 'label',
    'rawOrLabelHeaders': 'label',
    'exportCheckboxLabel': 'false',
    'exportSurveyFields': 'false',
    'exportDataAccessGroups': 'false',
    'returnFormat': 'json'
}

In [53]:
r = requests.post('https://svriglobal.org/redcap/api/', data=data)
print('HTTP Status:', r.status_code)
print(r.json())

HTTP Status: 200
[{'record': '56-1', 'redcap_event_name': 'Visita 1 - Screening', 'redcap_repeat_instrument': '', 'redcap_repeat_instance': '', 'field_name': 'form_medico_ex_fisico_alt', 'value': '171'}, {'record': '56-1', 'redcap_event_name': 'Visita 1 - Screening', 'redcap_repeat_instrument': '', 'redcap_repeat_instance': '', 'field_name': 'form_medico_ex_fisico_peso', 'value': '65.1'}, {'record': '56-1', 'redcap_event_name': 'Visita 1 - Screening', 'redcap_repeat_instrument': '', 'redcap_repeat_instance': '', 'field_name': 'form_medico_ex_fisico_q4', 'value': '22.26'}, {'record': '57-1', 'redcap_event_name': 'Visita 1 - Screening', 'redcap_repeat_instrument': '', 'redcap_repeat_instance': '', 'field_name': 'form_medico_ex_fisico_alt', 'value': '180'}, {'record': '57-1', 'redcap_event_name': 'Visita 1 - Screening', 'redcap_repeat_instrument': '', 'redcap_repeat_instance': '', 'field_name': 'form_medico_ex_fisico_peso', 'value': '80'}, {'record': '57-1', 'redcap_event_name': 'Visita 1

In [54]:
df_antropometria = pd.DataFrame(r.json())
df_antropometria.head()

,record,redcap_event_name,redcap_repeat_instrument,redcap_repeat_instance,field_name,value
0,56-1,Visita 1 - Screening,,,form_medico_ex_fisico_alt,171
1,56-1,Visita 1 - Screening,,,form_medico_ex_fisico_peso,65.1
2,56-1,Visita 1 - Screening,,,form_medico_ex_fisico_q4,22.26
3,57-1,Visita 1 - Screening,,,form_medico_ex_fisico_alt,180
4,57-1,Visita 1 - Screening,,,form_medico_ex_fisico_peso,80


In [55]:
field_name_list = [
    'form_medico_ex_fisico_peso',
    'form_medico_ex_fisico_alt',
    'form_medico_ex_fisico_q4'
]

df_antropometria_dt_filtered = df_antropometria[df_antropometria['field_name'].isin(field_name_list)]
df_antropometria_dt_filtered.drop(columns=['redcap_event_name', 'redcap_repeat_instrument', 'redcap_repeat_instance' ], inplace=True)
# drop duplicates
df_antropometria_dt_filtered = df_antropometria_dt_filtered.drop_duplicates()
#excluir a linha em que field_name == record_id
df_antropometria_dt_filtered = df_antropometria_dt_filtered[df_antropometria_dt_filtered['field_name'] != 'record_id']

df_antropometria_dt_filtered = df_antropometria_dt_filtered.pivot_table(
    index = ['record'],
    columns = 'field_name',
    values = 'value',
    aggfunc = 'first'
).reset_index()
df_antropometria_dt_filtered.rename(columns={
    'form_medico_ex_fisico_peso': 'fisio_peso',
    'form_medico_ex_fisico_alt': 'fisio_altura',
    'form_medico_ex_fisico_q4': 'fisio_imc'
}, inplace=True)
df_antropometria_dt_filtered.head()


field_name,record,fisio_altura,fisio_peso,fisio_imc
0,100-1,181,127.20,38.83
1,100-10,171,78,26.67
2,100-11,190,110,30.47
3,100-12,170,113,39.1
4,100-13,165,78.5,28.83


In [56]:
# tipificar as colunas como float
df_antropometria_dt_filtered['fisio_peso'] = df_antropometria_dt_filtered['fisio_peso'].astype(float)
df_antropometria_dt_filtered['fisio_altura'] = df_antropometria_dt_filtered['fisio_altura'].astype(float)
df_antropometria_dt_filtered['fisio_imc'] = df_antropometria_dt_filtered['fisio_imc'].astype(float)

In [57]:
# calcular Min, q1, median, q3, Max e plotar em tabela
colunas_alvo = ['fisio_peso', 'fisio_altura', 'fisio_imc']
tabela_resumo = df_antropometria_dt_filtered[colunas_alvo].copy()

for col in colunas_alvo:
    tabela_resumo[col] = pd.to_numeric(tabela_resumo[col], errors='coerce')

tabela_resumo = tabela_resumo.describe().loc[['min', '25%', '50%', '75%', 'max']]

tabela_resumo.index = ['Mínimo', 'Q1', 'Mediana', 'Q3', 'Máximo']

tabela_resumo.rename(columns={
    'fisio_peso': 'Peso',
    'fisio_altura': 'Altura',
    'fisio_imc': 'IMC'
}, inplace=True)
tabela_resumo = tabela_resumo.round(2)
tabela_resumo


field_name,Peso,Altura,IMC
Mínimo,50.8,1.55,15.00
Q1,74.0,168.00,24.16
Mediana,83.6,175.00,27.77
Q3,93.0,180.00,31.63
Máximo,127.2,191.00,347970.86


<h2>Boxplot com Jitter</h2>

In [58]:
fig = px.box(
    df_antropometria_dt_filtered,
    y='fisio_imc',
    labels = {'fisio_imc': 'IMC'}
    
)
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False)
)
fig.update_traces(
    marker=dict(color='blue'),
)

fig.show()

In [59]:
fig = px.box(
    df_antropometria_dt_filtered,
    y='fisio_peso',
    labels = {'fisio_peso': 'Peso'}
    
)
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False)
)
fig.update_traces(
    marker=dict(color='blue'),
    boxpoints='all'
)

fig.show()

In [60]:
fig = px.box(
    df_antropometria_dt_filtered,
    y='fisio_altura',
    labels = {'fisio_altura': 'altura'}
    
)
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False)
)
fig.update_traces(
    marker=dict(color='blue'),
    boxpoints='all'
)

fig.show()

<h2>Scatter Plot (Dispersão) Colorido</h2>

In [61]:
fig = px.scatter(
    data_frame = df_antropometria_dt_filtered,
    x='fisio_altura',
    y='fisio_peso',
    color='fisio_imc',  # Adiciona a cor baseada no IMC
    color_continuous_scale='Viridis'
)

fig.show()

<h2>Histograma com Marginal (Violin ou Box)</h2>

In [62]:
fig = px.histogram(
    data_frame=df_antropometria_dt_filtered,
    x='fisio_peso',
    nbins=10,
    labels={'fisio_peso': 'Peso'}
)

fig.update_layout(
    yaxis_title='Frequência',
    bargap=0,
    plot_bgcolor='white',
)
fig.update_traces(
    marker_line_width=1.5,
    marker_line_color='black'
)



fig.show()

In [63]:
fig = px.histogram(
    data_frame=df_antropometria_dt_filtered,
    x='fisio_altura',
    nbins=10,
    labels={'fisio_altura': 'altura'}
)

fig.update_layout(
    yaxis_title='Frequência',
    bargap=0,
    plot_bgcolor='white',
)
fig.update_traces(
    marker_line_width=1.5,
    marker_line_color='black'
)



fig.show()

In [64]:
fig = px.histogram(
    data_frame=df_antropometria_dt_filtered,
    x='fisio_imc',
    nbins=10,
    labels={'fisio_imc': 'imc'}
)

fig.update_layout(
    yaxis_title='Frequência',
    bargap=0,
    plot_bgcolor='white',
)
fig.update_traces(
    marker_line_width=1.5,
    marker_line_color='black'
)



fig.show()

<h1>FLOW DIAGRAM</h1>

In [130]:
data = {
    'token': token,
    'content': 'record',
    'action': 'export',
    'format': 'json',
    'type': 'eav',
    # 'records[]': id_list,  # Passa a lista diretamente aqui
    'fields[]': ['record_id'],
    'forms[0]': ['processo_de_consentimento'],
    'forms[1]': ['randomizao'],
    'forms[2]': ['formulrio_mdico'],
    'forms[3]': ['status_do_participante_de_pesquisa_no_estudo'],
    'forms[4]': ['demografia'],
    'events[0]': ['visita_1__screenin_arm_1'],
    'events[1]': ['visita_2__randomiz_arm_1'],
    'events[2]': ['status_do_particip_arm_1'],
    'events[3]': ['status_do_particip_arm_1'],
    'rawOrLabel': 'raw',
    'rawOrLabelHeaders': 'label',
    'exportCheckboxLabel': 'false',
    'exportSurveyFields': 'false',
    'exportDataAccessGroups': 'false',
    'returnFormat': 'json'
}
r = requests.post('https://svriglobal.org/redcap/api/', data=data)
print('HTTP Status:', r.status_code)
print(r.json())
df = pd.DataFrame(r.json())

HTTP Status: 200
[{'record': '56-1', 'redcap_event_name': 'status_do_particip_arm_1', 'redcap_repeat_instrument': '', 'redcap_repeat_instance': 1, 'field_name': 'record_id', 'value': '56-1'}, {'record': '56-1', 'redcap_event_name': 'status_do_particip_arm_1', 'redcap_repeat_instrument': '', 'redcap_repeat_instance': 1, 'field_name': 'status_instrumento', 'value': 'Status do Participante de Pesquisa no Estudo'}, {'record': '56-1', 'redcap_event_name': 'status_do_particip_arm_1', 'redcap_repeat_instrument': '', 'redcap_repeat_instance': 1, 'field_name': 'status_pp_dt_visita', 'value': '2025-12-19'}, {'record': '56-1', 'redcap_event_name': 'status_do_particip_arm_1', 'redcap_repeat_instrument': '', 'redcap_repeat_instance': 1, 'field_name': 'status_pp_status', 'value': '5'}, {'record': '56-1', 'redcap_event_name': 'status_do_particip_arm_1', 'redcap_repeat_instrument': '', 'redcap_repeat_instance': 1, 'field_name': 'status_pp_motivo', 'value': 'Solicitação do participante.'}, {'record': '

In [131]:
df_flow_diagram = pd.DataFrame(r.json())

In [132]:
df_flow_diagram.head()

,record,redcap_event_name,redcap_repeat_instrument,redcap_repeat_instance,field_name,value
0,56-1,status_do_particip_arm_1,,1,record_id,56-1
1,56-1,status_do_particip_arm_1,,1,status_instrumento,Status do Participante de Pesquisa no Estudo
2,56-1,status_do_particip_arm_1,,1,status_pp_dt_visita,2025-12-19
3,56-1,status_do_particip_arm_1,,1,status_pp_status,5
4,56-1,status_do_particip_arm_1,,1,status_pp_motivo,Solicitação do participante.


<h2>Total de participantes triados</h2>

In [133]:
df_triagem = df_flow_diagram.copy()
#contar valores da coluna record excluindo duplicatas
freq_triados = df_triagem['record'].nunique()
freq_triados

80

<h2>Total de participantes que assinaram o TCLE</h2>

In [134]:
assinaram_tcle = df_triagem.copy()
assinaram_tcle = assinaram_tcle[assinaram_tcle['field_name'] == 'form_medico_consentimento']

In [135]:
freq_assinaram_tcle = assinaram_tcle['record'].nunique()
freq_assinaram_tcle

78

<h2>Falhas de triagem</h2>

In [136]:
falhas_de_triagem = df_triagem.copy()
falhas_de_triagem = falhas_de_triagem[falhas_de_triagem['field_name'] == 'status_pp_status']
falhas_de_triagem.head()


,record,redcap_event_name,redcap_repeat_instrument,redcap_repeat_instance,field_name,value
3,56-1,status_do_particip_arm_1,,1,status_pp_status,5
147,57-1,status_do_particip_arm_1,,1,status_pp_status,2
406,78-1,status_do_particip_arm_1,,1,status_pp_status,2
670,78-2,status_do_particip_arm_1,,1,status_pp_status,2
926,78-3,status_do_particip_arm_1,,1,status_pp_status,5


In [137]:
#contagem total de records
num_records = falhas_de_triagem['record'].count()
print(f"Número total de records: {num_records}")



Número total de records: 81


In [138]:
# Drop de duplicatas de falhas_de_triagem por record
falhas_de_triagem.drop_duplicates(subset='record', inplace=True)

# contagem do número de records
num_records = falhas_de_triagem['record'].nunique()
print(f"Número de records únicos: {num_records}")

Número de records únicos: 80


In [139]:
mapping = {
    '1': 'Ativo',
    '2': 'Concluído',
    '3': 'Descontinuado',
    '4': 'Triagem',
    '5': 'Falha de seleção',
    '6': 'Retirada de TCLE',
    '7': 'Falha de Pré Triagem',
    '8': 'Morte',
    '9': 'Perda de seguimento'
}
# substitui os códigos pelos rótulos correspondentes
falhas_de_triagem['value'] = falhas_de_triagem['value'].map(mapping)
falhas_de_triagem.head()

,record,redcap_event_name,redcap_repeat_instrument,redcap_repeat_instance,field_name,value
3,56-1,status_do_particip_arm_1,,1,status_pp_status,Falha de seleção
147,57-1,status_do_particip_arm_1,,1,status_pp_status,Concluído
406,78-1,status_do_particip_arm_1,,1,status_pp_status,Concluído
670,78-2,status_do_particip_arm_1,,1,status_pp_status,Concluído
926,78-3,status_do_particip_arm_1,,1,status_pp_status,Falha de seleção


In [140]:
falhas_de_triagem=falhas_de_triagem[[
    'record',
    'value'
]]
falhas_de_triagem.rename(columns={
    'record': 'ID',
    'value': 'Status'
}, inplace=True)
falhas_de_triagem.head()


,ID,Status
3,56-1,Falha de seleção
147,57-1,Concluído
406,78-1,Concluído
670,78-2,Concluído
926,78-3,Falha de seleção


In [141]:
status=falhas_de_triagem.copy()
status = status['Status'].value_counts()
status

Status
Ativo                  33
Falha de seleção       17
Concluído              17
Triagem                 7
Perda de seguimento     4
Retirada de TCLE        2
Name: count, dtype: int64

In [142]:
status_relativo=falhas_de_triagem.copy()
status_relativo = round(status_relativo['Status'].value_counts(normalize=True)*100,2)
status_relativo

Status
Ativo                  41.25
Falha de seleção       21.25
Concluído              21.25
Triagem                 8.75
Perda de seguimento     5.00
Retirada de TCLE        2.50
Name: proportion, dtype: float64

In [143]:

status_formatados = [quebrar_texto(site, largura_max=18) for site in status.index]
    
fig = px.bar(
    status, 
    x=status_formatados, 
    y=status.values, 
    labels={'x':'', 'y':''}, 
    title='Distribuição de Status'
)
# adicionar o valor no lado externo superior da barra, limpar o fundo e tirar a grade
fig.update_traces(texttemplate='%{y}', textposition='inside')
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False)
)
#Retirar os valores do eixo Y (somente exibir os valores nas barras)
fig.update_yaxes(showticklabels=False)
fig.show()

<h2>Total de participantes randomizados</h2>

In [144]:
participantes_randomizados = df_triagem.copy()
lista_variaveis = ['demografia_sexo','randomizacao_randomizacao', 'randomizacao_dt_visita']
participantes_randomizados = participantes_randomizados[participantes_randomizados['field_name'].isin(lista_variaveis)]
participantes_randomizados = participantes_randomizados[['record', 'field_name', 'value']]
participantes_randomizados = participantes_randomizados.pivot(
    index='record',
    columns='field_name',
    values='value'
).reset_index()
participantes_randomizados.columns.name = None
participantes_randomizados.head()

,record,demografia_sexo,randomizacao_dt_visita,randomizacao_randomizacao
0,100-1,2,NaN,NaN
1,100-10,2,2026-06-11,2
2,100-11,2,2026-06-11,2
3,100-12,2,NaN,NaN
4,100-13,1,2026-07-03,2


In [145]:
participantes_randomizados = participantes_randomizados.rename(columns={
    'record': 'participante',
    'randomizacao_dt_visita': 'data da randomizacao',
    'randomizacao_randomizacao': 'Braço Randomizado',
    'demografia_sexo': 'Sexo'
})
participantes_randomizados.head()

,participante,Sexo,data da randomizacao,Braço Randomizado
0,100-1,2,NaN,NaN
1,100-10,2,2026-06-11,2
2,100-11,2,2026-06-11,2
3,100-12,2,NaN,NaN
4,100-13,1,2026-07-03,2


In [146]:
# excluir linhas em que 'data da randomizacao' ou 'Braço Randomizado' sejam nulos
participantes_randomizados = participantes_randomizados.dropna(subset=['Sexo','data da randomizacao', 'Braço Randomizado'])
participantes_randomizados.head()


,participante,Sexo,data da randomizacao,Braço Randomizado
1,100-10,2,2026-06-11,2
2,100-11,2,2026-06-11,2
4,100-13,1,2026-07-03,2
5,100-14,1,2026-07-24,2
6,100-15,2,2026-08-07,2


In [147]:
# tipificar os dados
participantes_randomizados['data da randomizacao'] = pd.to_datetime(participantes_randomizados['data da randomizacao'])
participantes_randomizados.head()

,participante,Sexo,data da randomizacao,Braço Randomizado
1,100-10,2,2026-06-11,2
2,100-11,2,2026-06-11,2
4,100-13,1,2026-07-03,2
5,100-14,1,2026-07-24,2
6,100-15,2,2026-08-07,2


In [148]:
#total de linhas em participantes_randomizados
participantes_randomizados.shape

(52, 4)

In [149]:
# excluir as duas datas mais recentes
participantes_randomizados = participantes_randomizados.sort_values(by='data da randomizacao')
participantes_randomizados = participantes_randomizados.iloc[:-2]
participantes_randomizados.shape

(50, 4)

In [150]:
mapping = {
    '1': 'Oxandrolona',
    '2': 'Placebo'
}
participantes_randomizados['Braço Randomizado'] = participantes_randomizados['Braço Randomizado'].map(mapping)
participantes_randomizados


,participante,Sexo,data da randomizacao,Braço Randomizado
56,57-1,2,2025-12-26,Placebo
65,79-1,1,2026-01-23,Oxandrolona
57,78-1,2,2026-01-26,Placebo
58,78-2,2,2026-01-27,Oxandrolona
60,78-4,2,2026-01-28,Oxandrolona
62,78-6,2,2026-01-29,Placebo
69,79-3,1,2026-02-10,Placebo
70,79-4,2,2026-02-13,Oxandrolona
72,79-6,2,2026-02-24,Oxandrolona
73,79-7,2,2026-02-26,Placebo


In [151]:
mapping_sexo = {
    '2': 'Masculino',
    '1': 'Feminino'
}
participantes_randomizados['Sexo'] = participantes_randomizados['Sexo'].map(mapping_sexo)
participantes_randomizados

,participante,Sexo,data da randomizacao,Braço Randomizado
56,57-1,Masculino,2025-12-26,Placebo
65,79-1,Feminino,2026-01-23,Oxandrolona
57,78-1,Masculino,2026-01-26,Placebo
58,78-2,Masculino,2026-01-27,Oxandrolona
60,78-4,Masculino,2026-01-28,Oxandrolona
62,78-6,Masculino,2026-01-29,Placebo
69,79-3,Feminino,2026-02-10,Placebo
70,79-4,Masculino,2026-02-13,Oxandrolona
72,79-6,Masculino,2026-02-24,Oxandrolona
73,79-7,Masculino,2026-02-26,Placebo


In [152]:
# Total de participantes randomizados
freq_participantes_randomizados = len(participantes_randomizados)
print (f" participantes randomizado: {freq_participantes_randomizados}")

 participantes randomizado: 50


<h2>Participantes randomizados por braço</h2>

In [154]:
participantes_randomizados_por_braco = participantes_randomizados.copy()
participantes_randomizados_por_braco = participantes_randomizados_por_braco['Braço Randomizado'].value_counts()
participantes_randomizados_por_braco

Braço Randomizado
Oxandrolona    39
Placebo        11
Name: count, dtype: int64

In [155]:
participantes_randomizados_por_braco_relativo = participantes_randomizados.copy()
participantes_randomizados_por_braco_relativo = round(participantes_randomizados_por_braco_relativo['Braço Randomizado'].value_counts(normalize=True)*100,2)
participantes_randomizados_por_braco_relativo

Braço Randomizado
Oxandrolona    78.0
Placebo        22.0
Name: proportion, dtype: float64

In [156]:
# Participantes randomizados (valor absoluto) por Braço por sexo
participantes_randomizados.groupby(['Braço Randomizado', 'Sexo']).size().unstack(fill_value=0)

Sexo,Feminino,Masculino
Braço Randomizado,,
Oxandrolona,5,34
Placebo,3,8


In [157]:
# Participantes randomizados (valor relativo) por Braço por sexo
(participantes_randomizados.groupby(['Braço Randomizado', 'Sexo']).size().unstack(fill_value=0).apply(lambda x: x / x.sum(), axis=1)*100).round(2)

Sexo,Feminino,Masculino
Braço Randomizado,,
Oxandrolona,12.82,87.18
Placebo,27.27,72.73


In [159]:
alocacao_pk = df_triagem.copy()
alocacao_pk = alocacao_pk[alocacao_pk['field_name']=='randomizacao_alocacao_pk']
alocacao_pk = alocacao_pk[['record', 'value']]
alocacao_pk = alocacao_pk.rename(columns={'record': 'Record', 'value': 'Alocacao PK'})
mapping = {
    '1': 'Não alocado',
    '2': 'Alocado'
}
alocacao_pk['Alocacao PK'] = alocacao_pk['Alocacao PK'].map(mapping)
alocacao_pk.head()


,Record,Alocacao PK
398,57-1,Alocado
659,78-1,Alocado
814,78-2,Alocado
1735,78-6,Alocado
2036,78-8,Alocado


In [160]:
freq_alocacao_pk = alocacao_pk['Alocacao PK'].value_counts()
freq_alocacao_pk

Alocacao PK
Alocado    22
Name: count, dtype: int64